In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report

# 1. Load and clean
dataset = pd.read_csv('/content/reddit_preprocessing.csv')
cleaned_dataset = dataset.dropna()

X_cleaned = cleaned_dataset['clean_comment']
Y_cleaned = cleaned_dataset['category']

# 2. Train/Test Split
X_train_cleaned, X_test_cleaned, Y_train_cleaned, Y_test_cleaned = train_test_split(
    X_cleaned,
    Y_cleaned,
    test_size=0.2,
    random_state=42,
    stratify=Y_cleaned # Ensures train and test sets have the same class distribution
)

# 3. Vectorize
tfidf_cleaned = TfidfVectorizer(ngram_range=(1, 3), max_features=10000)
X_train_tfidf_cleaned = tfidf_cleaned.fit_transform(X_train_cleaned)
X_test_tfidf_cleaned = tfidf_cleaned.transform(X_test_cleaned) # Fixed typo here

# 4. Define Base Models (with n_jobs=-1 for max speed)
lightgbm_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    metric="multi_logloss",
    class_weight="balanced", # Removed duplicate is_unbalance parameter
    reg_alpha=0.1,
    reg_lambda=0.1,
    n_estimators=367,
    max_depth=20,
    n_jobs=-1
)

logreg_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',
    multi_class='multinomial',
    n_jobs=-1
)

# 5. Define Meta-Learner (Standardized to Logistic Regression)
meta_learner = LogisticRegression(n_jobs=-1)

# 6. Build and Train Stacking Classifier
stacking_model = StackingClassifier(
    estimators=[
        ('lightgbm', lightgbm_model),
        ('logistic_regression', logreg_model),
    ],
    final_estimator=meta_learner,
    cv=5,
    n_jobs=-1 # Forces 5-fold CV to train in parallel
)

# Train and Predict
stacking_model.fit(X_train_tfidf_cleaned, Y_train_cleaned)
Y_pred = stacking_model.predict(X_test_tfidf_cleaned)

# Evaluate
print(classification_report(Y_test_cleaned, Y_pred))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

          -1       0.82      0.75      0.78      1650
           0       0.86      0.95      0.91      2529
           1       0.90      0.87      0.88      3154

    accuracy                           0.87      7333
   macro avg       0.86      0.86      0.86      7333
weighted avg       0.87      0.87      0.87      7333

